# TEKNOFEST Healthcare AI — Phase 01: Exploratory Data Analysis
## Missense Variant Pathogenicity Classification

**Objective**: Understand the dataset deeply before any modeling.

**Datasets**:
- MASTER: General training set
- CFTR: Cystic Fibrosis gene panel
- PAH: Phenylalanine Hydroxylase gene panel
- KANSER: Cancer gene panel


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
%matplotlib inline

print("Libraries loaded successfully.")


## A. Dataset Loading

In [ ]:
from pathlib import Path
from collections import OrderedDict

DATA_DIR = Path("EĞİTİM (TRAIN) SETLERİ 2")

datasets = OrderedDict()
for fname in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(fname)
    key = fname.stem.replace("YARISMA_TRAIN_", "")
    datasets[key] = df
    print(f"{key}: {df.shape[0]} rows x {df.shape[1]} cols — {fname.stat().st_size / 1024:.0f} KB")

master = datasets["MASTER"]


## B. Schema Inspection

In [ ]:
al_cols = [c for c in master.columns if c.startswith("AL_")]
cat_cols = [c for c in master.columns if c.startswith("CAT_")]
ek_cols = [c for c in master.columns if c.startswith("EK_")]
aa_cols = [c for c in master.columns if c.startswith("AA_")]

print(f"AL features: {len(al_cols)}")
print(f"CAT features: {len(cat_cols)}")
print(f"EK features: {len(ek_cols)}")
print(f"AA features: {len(aa_cols)}")
print(f"Other: Variant_ID, Label")
print(f"\nTotal columns: {master.shape[1]}")

print("\n--- Data Types ---")
print(master.dtypes.value_counts())

print("\n--- Constant columns per dataset ---")
for key, df in datasets.items():
    const = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
    print(f"  {key}: {len(const)} constant columns")

print("\n--- Duplicate rows per dataset ---")
for key, df in datasets.items():
    print(f"  {key}: {df.duplicated().sum()} duplicate rows")


In [ ]:
print("\n--- Schema overview (first 5 + last 5 columns) ---")
schema = pd.DataFrame({
    "Column": master.columns,
    "Dtype": master.dtypes.astype(str),
    "Non-Null": master.notna().sum(),
    "Missing%": (master.isnull().mean() * 100).round(2),
    "Unique": master.nunique()
})
display(pd.concat([schema.head(10), schema.tail(10)]))


## C. Target Label Analysis

In [ ]:
class_dist = []
for key, df in datasets.items():
    vc = df["Label"].value_counts()
    total = len(df)
    n_path = int(vc.get(1, 0))
    n_ben = int(vc.get(0, 0))
    pos_ratio = n_path / total
    imb_ratio = max(n_path, n_ben) / min(n_path, n_ben) if min(n_path, n_ben) > 0 else float('inf')
    class_dist.append({
        "Dataset": key, "Total": total,
        "Pathogenic (1)": n_path, "Benign (0)": n_ben,
        "Positive Ratio": round(pos_ratio, 4),
        "Imbalance Ratio": round(imb_ratio, 2)
    })

class_dist_df = pd.DataFrame(class_dist)
display(class_dist_df)


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for idx, (key, df) in enumerate(datasets.items()):
    vc = df["Label"].value_counts().sort_index()
    axes[idx].bar(["Benign (0)", "Pathogenic (1)"], [vc.get(0, 0), vc.get(1, 0)],
                  color=["#2196F3", "#F44336"])
    axes[idx].set_title(f"{key} (n={len(df)})")
    axes[idx].set_ylabel("Count")
    for j, v in enumerate([vc.get(0, 0), vc.get(1, 0)]):
        axes[idx].text(j, v + 2, str(v), ha='center', fontsize=10)
plt.suptitle("Class Distribution Across Datasets", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## D. Missing Value Analysis

In [ ]:
for key, df in datasets.items():
    miss_pct = df.isnull().mean() * 100
    bands = {
        "0%": (miss_pct == 0).sum(),
        "0-5%": ((miss_pct > 0) & (miss_pct <= 5)).sum(),
        "5-20%": ((miss_pct > 5) & (miss_pct <= 20)).sum(),
        "20-50%": ((miss_pct > 20) & (miss_pct <= 50)).sum(),
        "50-80%": ((miss_pct > 50) & (miss_pct <= 80)).sum(),
        "80-95%": ((miss_pct > 80) & (miss_pct <= 95)).sum(),
        ">95%": (miss_pct > 95).sum()
    }
    print(f"\n--- {key} ---")
    for band, cnt in bands.items():
        print(f"  {band}: {cnt} columns")

    row_miss = df.isnull().mean(axis=1) * 100
    print(f"  Row missingness: mean={row_miss.mean():.1f}%, median={row_miss.median():.1f}%, max={row_miss.max():.1f}%")


In [ ]:
# Top 50 most-missing features in MASTER
miss_pct_master = master[al_cols + ek_cols].isnull().mean().sort_values(ascending=False)
top50 = miss_pct_master.head(50)

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(len(top50)), top50.values * 100, color='#FF9800')
ax.set_yticks(range(len(top50)))
ax.set_yticklabels(top50.index, fontsize=7)
ax.set_xlabel("Missing %")
ax.set_title("Top 50 Features by Missing Percentage (MASTER)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Differential missingness by target
path_mask = master["Label"] == 1
ben_mask = master["Label"] == 0

diff_miss = []
for col in al_cols + ek_cols:
    miss_path = master.loc[path_mask, col].isnull().mean()
    miss_ben = master.loc[ben_mask, col].isnull().mean()
    diff_miss.append({
        "Column": col,
        "Miss% Pathogenic": round(miss_path * 100, 2),
        "Miss% Benign": round(miss_ben * 100, 2),
        "Abs Diff": round(abs(miss_path - miss_ben) * 100, 2)
    })

diff_miss_df = pd.DataFrame(diff_miss).sort_values("Abs Diff", ascending=False)
print("Top 15 features with differential missingness:")
display(diff_miss_df.head(15))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for key, df in datasets.items():
    miss_profile = df[al_cols].isnull().mean().sort_values()
    ax.plot(range(len(miss_profile)), miss_profile.values * 100, label=key, alpha=0.8)
ax.set_xlabel("Feature Index (sorted by missingness)")
ax.set_ylabel("Missing %")
ax.set_title("Missingness Profile Across Datasets")
ax.legend()
plt.tight_layout()
plt.show()


## E. Distribution and Outlier Analysis

In [ ]:
numeric_feats = [c for c in al_cols + ek_cols
                 if c in master.columns and master[c].dtype in [np.float64, np.int64, float, int]]

desc_stats = []
for col in numeric_feats:
    s = master[col].dropna()
    if len(s) == 0:
        continue
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((s < q1 - 3 * iqr) | (s > q3 + 3 * iqr)).sum()

    feat_type = "numeric"
    if s.min() >= 0 and s.max() <= 1.001:
        feat_type = "probability/score [0,1]"
    elif s.nunique() == 2:
        feat_type = "binary"
    elif s.min() >= 0 and s.max() > 1000:
        feat_type = "count/position-like"

    desc_stats.append({
        "Column": col,
        "Count": len(s),
        "Missing%": round(master[col].isnull().mean()*100, 2),
        "Mean": round(s.mean(), 6),
        "Std": round(s.std(), 6),
        "Min": round(s.min(), 6),
        "Max": round(s.max(), 6),
        "Skewness": round(float(s.skew()), 4),
        "Inferred Type": feat_type
    })

desc_df = pd.DataFrame(desc_stats)
print("Feature type breakdown:")
display(desc_df["Inferred Type"].value_counts().to_frame("Count"))
print(f"\nHighly skewed (|skew|>3): {(desc_df['Skewness'].abs() > 3).sum()}")


In [ ]:
# Feature type pie chart
type_counts = desc_df["Inferred Type"].value_counts()
fig, ax = plt.subplots(figsize=(8, 6))
ax.pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
       colors=['#4CAF50', '#2196F3', '#FF9800', '#9C27B0'])
ax.set_title("Feature Type Distribution (MASTER)")
plt.tight_layout()
plt.show()


## F. Feature-Target Association

In [ ]:
path_idx = master["Label"] == 1
ben_idx = master["Label"] == 0

assoc_results = []
for col in numeric_feats:
    s_path = master.loc[path_idx, col].dropna()
    s_ben = master.loc[ben_idx, col].dropna()
    if len(s_path) < 5 or len(s_ben) < 5:
        continue
    med_diff = s_path.median() - s_ben.median()
    try:
        u_stat, p_val = mannwhitneyu(s_path, s_ben, alternative='two-sided')
        n1, n2 = len(s_path), len(s_ben)
        effect_size = 1 - (2 * u_stat) / (n1 * n2)
    except:
        p_val, effect_size = np.nan, np.nan

    assoc_results.append({
        "Column": col,
        "Med Pathogenic": round(float(s_path.median()), 6),
        "Med Benign": round(float(s_ben.median()), 6),
        "Med Diff": round(float(med_diff), 6),
        "p-value": float(p_val),
        "Effect Size (r)": round(float(effect_size), 4),
        "Abs Effect": round(abs(float(effect_size)), 4)
    })

assoc_df = pd.DataFrame(assoc_results).sort_values("Abs Effect", ascending=False)
print(f"Total features tested: {len(assoc_df)}")
print(f"Significant (p<0.001): {(assoc_df['p-value'] < 0.001).sum()}")
print(f"Large effect (|r|>0.5): {(assoc_df['Abs Effect'] > 0.5).sum()}")
print("\nTop 20 features:")
display(assoc_df.head(20))


In [ ]:
# Top 20 barplot
top20 = assoc_df.head(20)
fig, ax = plt.subplots(figsize=(12, 7))
colors = ['#F44336' if d > 0 else '#2196F3' for d in top20["Effect Size (r)"].values]
ax.barh(range(len(top20)), top20["Abs Effect"].values, color=colors)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20["Column"].values, fontsize=9)
ax.set_xlabel("Absolute Effect Size (rank-biserial r)")
ax.set_title("Top 20 Features by Class Separation")
ax.invert_yaxis()
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#F44336', label='Higher in Pathogenic'),
                   Patch(facecolor='#2196F3', label='Higher in Benign')]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()


## G. Correlation and Redundancy

In [ ]:
# Correlation heatmap of top 30 features
top30_feats = assoc_df.head(30)["Column"].tolist()
top30_avail = [f for f in top30_feats if f in master.columns]

corr_sub = master[top30_avail].corr(method='spearman')
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_sub, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=True, yticklabels=True, ax=ax, square=True, linewidths=0.5)
ax.set_title("Spearman Correlation: Top 30 Predictive Features")
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# Count highly correlated pairs
feat_for_corr = [c for c in numeric_feats if master[c].notna().sum() > 100]
corr_full = master[feat_for_corr].corr(method='spearman')

pairs_85 = 0
pairs_95 = 0
for i in range(len(feat_for_corr)):
    for j in range(i+1, len(feat_for_corr)):
        r = abs(corr_full.iloc[i, j])
        if r > 0.95:
            pairs_95 += 1
        if r > 0.85:
            pairs_85 += 1

print(f"Pairs with |r| > 0.85: {pairs_85}")
print(f"Pairs with |r| > 0.95: {pairs_95}")


## H. Panel Comparison

In [ ]:
panel_comp = []
for key, df in datasets.items():
    n_path = (df["Label"] == 1).sum()
    n_ben = (df["Label"] == 0).sum()
    miss_al = df[al_cols].isnull().mean().mean() * 100
    n_const = sum(df[c].nunique(dropna=True) <= 1 for c in df.columns)
    panel_comp.append({
        "Dataset": key,
        "Rows": len(df),
        "Pathogenic": n_path,
        "Benign": n_ben,
        "Pos Ratio": round(n_path/len(df), 3),
        "Mean Missing% (AL)": round(miss_al, 1),
        "Constant Cols": n_const
    })

display(pd.DataFrame(panel_comp))

# Variant_ID overlaps
print("\nVariant_ID Overlap Matrix:")
overlap = {}
for k1, d1 in datasets.items():
    overlap[k1] = {}
    for k2, d2 in datasets.items():
        overlap[k1][k2] = len(set(d1["Variant_ID"]) & set(d2["Variant_ID"]))
display(pd.DataFrame(overlap))


## EK Features Deep Dive (Leakage Check)

In [ ]:
ek_numeric = [c for c in ek_cols if c in master.columns and master[c].dtype in [np.float64, float]]
n_ek = len(ek_numeric)
if n_ek > 0:
    fig, axes = plt.subplots(1, min(n_ek, 6), figsize=(4 * min(n_ek, 6), 4))
    if n_ek == 1:
        axes = [axes]
    for idx, col in enumerate(ek_numeric[:6]):
        ax = axes[idx]
        master[master["Label"]==0][col].dropna().plot(kind='kde', ax=ax, label='Benign', color='#2196F3')
        master[master["Label"]==1][col].dropna().plot(kind='kde', ax=ax, label='Pathogenic', color='#F44336')
        ax.set_title(col)
        ax.legend()
    plt.suptitle("EK Feature Distributions by Class", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

for col in ek_numeric:
    s = master[col].dropna()
    print(f"{col}: range [{s.min():.3f}, {s.max():.3f}], mean={s.mean():.3f}, missing={master[col].isnull().mean()*100:.1f}%")


## I. Leakage Risk Register

In [ ]:
leakage_risks = pd.read_csv("reports/phase_01_data_understanding/leakage_risk_register.csv")
display(leakage_risks[["Risk", "Severity", "Mitigation"]])


## J. Phase 2 Recommendations

### Baseline Model
- **LightGBM or XGBoost** with default hyperparameters
- Native missing value handling

### Validation
- **Stratified 5-Fold CV** on MASTER
- Variant_ID-aware splitting

### Metrics
- **Primary**: ROC-AUC
- **Secondary**: PR-AUC, Sensitivity, F1
- Sensitivity-focused thresholding

### Strategy
1. Train global model on MASTER
2. Evaluate per-panel performance
3. Build models WITH and WITHOUT EK features
4. If panel performance is poor: fine-tune panel-specific models

### Risks to Monitor
1. EK features may cause circular prediction
2. Small panel sizes = high variance
3. Distribution shift between MASTER and panels
4. Anonymized features prevent validation
5. Differential missingness may confound results


## Categorical Feature Analysis

In [ ]:
for col in cat_cols + aa_cols:
    print(f"\n--- {col} ---")
    print(f"  Unique values: {master[col].nunique(dropna=True)}")
    print(f"  Missing: {master[col].isnull().mean()*100:.1f}%")
    print(f"  Sample values: {list(master[col].dropna().unique()[:10])}")
    print(f"  Value counts (top 5):")
    print(master[col].value_counts().head(5).to_string())
